# TOP ошибок и выборка ИНН по ФП/СФП

Аналитика построена по логике отчета `analysis_scenario_ipu_results.ipynb`:
- дедупликация карточек по двум потокам (scenario / ipu),
- выделение ошибок на этапе сценариев и ИПУ,
- TOP-10 по ошибкам в разрезе сегментов,
- выборка до 100 уникальных ИНН для TOP-5 в каждом сегменте,
- экспорт в Excel (каждая секция на отдельном листе).

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
ALLOWED_SOURCES = {
    "Н2.0", "H2.0", "H2O", "Н2О",
    "Справочник CRM-системы", "CRM-система", "CRM", "DIASOFT",
}
DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]
SCR_DATE_COLS = ["END_DATE_SCR_FCT", "END_DATE_SCR_PLAN"]
IPU_DATE_COLS = ["FIRST_END_DATE_EVENT", "NEW_PLAN_END_DATE_EVT", "END_EVENT_DATE_FACT"]

SCR_ERROR_TOKENS = [
    "информация проверена",
    "отсутствуют основания для выявления",
]
IPU_ERROR_TOKENS = [
    "срок окончания плана истек",
    "результат реализации по состоянию",
    "отсутствует",
]


_LAT2CYR = str.maketrans({
    "A": "А", "B": "В", "C": "С", "E": "Е", "H": "Н", "K": "К",
    "M": "М", "O": "О", "P": "Р", "T": "Т", "X": "Х",
    "a": "а", "c": "с", "e": "е", "o": "о", "p": "р", "x": "х", "y": "у",
})


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


def norm_text(value):
    if pd.isna(value) or value == "":
        return ""
    s = str(value)
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(_LAT2CYR)
    s = re.sub(r"[\s\xa0\u200b\ufeff]+", " ", s)
    return s.strip()


def normalize_source(value: str) -> str:
    if pd.isna(value):
        return ""
    src = norm_text(value).upper()
    return src.replace(" ", "")


def has_all_tokens(text: str, tokens: list[str]) -> bool:
    base = norm_text(text).lower()
    return all(token in base for token in tokens)


def is_scenario_error(text: str) -> bool:
    return has_all_tokens(text, SCR_ERROR_TOKENS)


def is_ipu_error(text: str) -> bool:
    return has_all_tokens(text, IPU_ERROR_TOKENS)


# Жесткие банковые пути (без fallback).
DATA_DIR = Path('/home/jovyan/documents/DMUKP_FP_DEF/data')
CRM_FILE = DATA_DIR / 'crm_last_version.csv'
EXCEL_PATH = DATA_DIR / 'error_top_inn_by_segment.xlsx'

print(f"CRM_FILE: {CRM_FILE}")
print(f"EXCEL_PATH: {EXCEL_PATH}")

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
if len(raw.columns) == 1 and ";" in str(raw.columns[0]):
    raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False, sep=";")
raw.columns = raw.columns.str.strip()

if "DESC_TEXT.1" in raw.columns and "DESC_TEXT_1" not in raw.columns:
    raw = raw.rename(columns={"DESC_TEXT.1": "DESC_TEXT_1"})

if "VAL" in raw.columns:
    source_norm = raw["VAL"].apply(normalize_source)
    allowed_norm = {normalize_source(v) for v in ALLOWED_SOURCES}
    filtered = raw[source_norm.isin(allowed_norm)].copy()
    # Если в локальной выгрузке источники отличаются от отчета, не обнуляем выборку.
    raw = filtered if len(filtered) > 0 else raw.copy()

# Поддержка как «сырых» CRM-колонок, так и уже подготовленного набора.
inn_col = "X_INN" if "X_INN" in raw.columns else "inn"
segment_col = "X_AREA_RESP" if "X_AREA_RESP" in raw.columns else "segment"
fp_num_col = "NUMBER_FP_SFP" if "NUMBER_FP_SFP" in raw.columns else "fp_num"
fp_type_col = "TYPE_FP" if "TYPE_FP" in raw.columns else "fp_type"

raw["inn"] = raw[inn_col].apply(normalize_inn)
raw["segment"] = raw[segment_col].astype(str).str.strip().map(SEGMENT_MAP).fillna(raw[segment_col].astype(str).str.strip())
raw["fp_num"] = raw[fp_num_col].astype(str).str.strip()
raw["fp_type"] = raw[fp_type_col].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})

raw = raw.dropna(subset=["inn", "fp_num"]).copy()
raw = raw[raw["segment"].notna()].copy()

raw["scenario"] = raw["DESC_TEXT_1"].apply(norm_text)
raw["ipu"] = raw["DESC_TEXT"].apply(norm_text)

for col in SCR_DATE_COLS + IPU_DATE_COLS:
    raw[f"_{col}_dt"] = pd.to_datetime(raw[col], dayfirst=True, errors="coerce")

scr_dt_cols = [f"_{col}_dt" for col in SCR_DATE_COLS]
ipu_dt_cols = [f"_{col}_dt" for col in IPU_DATE_COLS]

raw["_scr_max"] = raw[scr_dt_cols].max(axis=1)
raw["_ipu_max"] = raw[ipu_dt_cols].max(axis=1)

# Делим на потоки по наличию дат ИПУ внутри карточки.
grp_has_ipu = raw.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()
stream1_raw = raw[~grp_has_ipu].copy()
stream2_raw = raw[grp_has_ipu].copy()

stream1_raw = stream1_raw[stream1_raw.groupby(DEDUP_KEY)["_scr_max"].transform("max").notna()]
stream1_sorted = stream1_raw.sort_values(["_END_DATE_SCR_FCT_dt", "_END_DATE_SCR_PLAN_dt"], ascending=[False, False], na_position="last")
df_stream1 = stream1_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream1["stream"] = "scenario"

stream2_raw = stream2_raw[stream2_raw.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()]
stream2_sorted = stream2_raw.sort_values(["_END_EVENT_DATE_FACT_dt", "_NEW_PLAN_END_DATE_EVT_dt", "_FIRST_END_DATE_EVENT_dt"], ascending=[False, False, False], na_position="last")
df_stream2 = stream2_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream2["stream"] = "ipu"

df = pd.concat([df_stream1, df_stream2], ignore_index=True)

# Выделяем ошибки по текстовым токенам (устойчиво к префиксам/вариантам формулировок).
df["scr_class"] = df["scenario"].apply(lambda x: "ошибка" if is_scenario_error(x) else "прочее")
df["ipu_class"] = df["ipu"].apply(lambda x: "ошибка" if is_ipu_error(x) else "прочее")

print(f"raw rows: {len(raw):,}")
print(f"dedup rows (df): {len(df):,}")
print(df[["segment", "fp_num", "fp_type", "stream", "scr_class", "ipu_class"]].head())

In [ ]:
def top_by_segment_error(stage_df: pd.DataFrame, class_col: str, top_n: int = 10) -> pd.DataFrame:
    error_rows = stage_df[stage_df[class_col] == "ошибка"].copy()
    if error_rows.empty:
        return pd.DataFrame(columns=["segment", "rank", "fp_num", "fp_type", "error_count"])

    agg = (
        error_rows
        .groupby(["segment", "fp_num", "fp_type"], as_index=False)
        .size()
        .rename(columns={"size": "error_count"})
    )

    agg = agg.sort_values(["segment", "error_count", "fp_num", "fp_type"], ascending=[True, False, True, True])
    agg["rank"] = agg.groupby("segment").cumcount() + 1
    return agg[agg["rank"] <= top_n].reset_index(drop=True)


scenario_stage = df.copy()
ipu_stage = df[df["stream"] == "ipu"].copy()

top10_scenario_error = top_by_segment_error(scenario_stage, class_col="scr_class", top_n=10)
top10_ipu_error = top_by_segment_error(ipu_stage, class_col="ipu_class", top_n=10)

print("TOP-10 по ошибкам сценариев")
display(top10_scenario_error)
print("TOP-10 по ошибкам ИПУ")
display(top10_ipu_error)

In [ ]:
def top5_inn_lists(stage_df: pd.DataFrame, top_df: pd.DataFrame, class_col: str, stage_name: str, inn_limit: int = 100) -> pd.DataFrame:
    top5 = top_df[top_df["rank"] <= 5].copy()
    rows = []

    for item in top5.itertuples(index=False):
        sub = stage_df[
            (stage_df["segment"] == item.segment)
            & (stage_df["fp_num"] == item.fp_num)
            & (stage_df["fp_type"] == item.fp_type)
            & (stage_df[class_col] == "ошибка")
        ].copy()

        inn_sample = sub["inn"].dropna().drop_duplicates().head(inn_limit)
        for idx, inn in enumerate(inn_sample, start=1):
            rows.append(
                {
                    "analysis_stage": stage_name,
                    "segment": item.segment,
                    "rank": int(item.rank),
                    "fp_num": item.fp_num,
                    "fp_type": item.fp_type,
                    "error_count": int(item.error_count),
                    "inn_position": idx,
                    "inn": inn,
                }
            )

    return pd.DataFrame(rows)


top5_scenario_inn100 = top5_inn_lists(
    stage_df=scenario_stage,
    top_df=top10_scenario_error,
    class_col="scr_class",
    stage_name="scenario",
    inn_limit=100,
)

top5_ipu_inn100 = top5_inn_lists(
    stage_df=ipu_stage,
    top_df=top10_ipu_error,
    class_col="ipu_class",
    stage_name="ipu",
    inn_limit=100,
)

print("ИНН по TOP-5 ошибок сценариев")
display(top5_scenario_inn100.head(20))
print("ИНН по TOP-5 ошибок ИПУ")
display(top5_ipu_inn100.head(20))

In [ ]:
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    top10_scenario_error.to_excel(writer, sheet_name="TOP10_Scenario_Error", index=False)
    top10_ipu_error.to_excel(writer, sheet_name="TOP10_IPU_Error", index=False)
    top5_scenario_inn100.to_excel(writer, sheet_name="TOP5_Scenario_INN100", index=False)
    top5_ipu_inn100.to_excel(writer, sheet_name="TOP5_IPU_INN100", index=False)

print(f"Excel сохранен: {EXCEL_PATH}")

# Контрольные проверки
scenario_per_segment = top10_scenario_error.groupby("segment")["rank"].max().rename("max_rank")
ipu_per_segment = top10_ipu_error.groupby("segment")["rank"].max().rename("max_rank")
print("\nПроверка TOP-10 (max rank по сегментам):")
print("Scenario:\n", scenario_per_segment)
print("IPU:\n", ipu_per_segment)

if not top5_scenario_inn100.empty:
    chk1 = top5_scenario_inn100.groupby(["segment", "fp_num", "fp_type"])["inn"].nunique().max()
else:
    chk1 = 0
if not top5_ipu_inn100.empty:
    chk2 = top5_ipu_inn100.groupby(["segment", "fp_num", "fp_type"])["inn"].nunique().max()
else:
    chk2 = 0

print("\nПроверка лимита ИНН (должно быть <=100):")
print("Scenario max unique INN:", chk1)
print("IPU max unique INN:", chk2)